In [1]:
import pandas as pd
from collections import Counter

In [46]:
# 원본 데이터
snli_test = pd.read_json('/data/hyeryung/mucoco/data/nli/snli_1.0/snli_1.0_test.jsonl',lines=True)
# EPR 데이터
epr_snli = pd.read_json('/data/hyeryung/mucoco/new_module/data/EPR/text_file/snli_annotation/snli_locate_labels.jsonl', lines=True)

# 저장 경로 1
raw_save_path = '/data/hyeryung/mucoco/new_module/data/NLI_locate/snli_test_contra_100_seed_999.jsonl'
templated_save_path = '/data/hyeryung/mucoco/new_module/data/NLI_locate/snli_test_contra_labelling_sheet.xlsx'

In [47]:
# ---------------------- 라벨링 대상 추출, 저장 ---------------------- #
# 원본에서 EPR에 없는 데이터만 추출
print(snli_test.shape[0])
snli_test = snli_test.loc[~snli_test['pairID'].isin(epr_snli['pairID']),:].copy()
print(snli_test.shape[0])
# contradiction만 추출
print(snli_test.shape[0])
snli_test = snli_test.loc[snli_test['gold_label']=='contradiction',:].copy()
print(snli_test.shape[0])
print(snli_test.gold_label.value_counts())
print(snli_test.annotator_labels.apply(Counter).value_counts())
# random 100개 추출
snli_test = snli_test.sample(100, random_state=999)
snli_test.to_json(raw_save_path, orient='records', lines=True)    
# double check
print(snli_test.gold_label.value_counts())

10000
9900
9900
3205
gold_label
contradiction    3205
Name: count, dtype: int64
annotator_labels
{'contradiction': 5}                                   2130
{'contradiction': 4, 'neutral': 1}                      531
{'contradiction': 4, 'entailment': 1}                   227
{'contradiction': 3, 'neutral': 2}                      198
{'contradiction': 3, 'neutral': 1, 'entailment': 1}      89
{'contradiction': 3, 'entailment': 2}                    29
{'contradiction': 4}                                      1
Name: count, dtype: int64
gold_label
contradiction    100
Name: count, dtype: int64


In [118]:
# ---------------------- 라벨링 형식으로 바꿔서 저장 ---------------------- #
# labelling 하는 곳 추가
snli_test = snli_test[['pairID', 'sentence1','sentence2']].copy()
# text를 단어로 분리 
snli_test.loc[:, 'sentence1_words'] = snli_test['sentence1'].str.split()
snli_test.loc[:, 'sentence2_words'] = snli_test['sentence2'].str.split()
max_len =max(snli_test['sentence1_words'].apply(len).max(),snli_test['sentence2_words'].apply(len).max()) 
print(max_len)
# padding
snli_test.loc[:, 'sentence1_words'] = snli_test['sentence1_words'].apply(lambda x: x + ['']*(max_len-len(x)))
snli_test.loc[:, 'sentence2_words'] = snli_test['sentence2_words'].apply(lambda x: x + ['']*(max_len-len(x)))
assert min(snli_test['sentence1_words'].apply(len).min(),snli_test['sentence2_words'].apply(len).min()) == max_len
# labelling 할 곳 추가
snli_test.loc[:, 'sentence1_labels'] = snli_test['sentence1_words'].apply(lambda x: ['']*max_len)
snli_test.loc[:, 'sentence2_labels'] = snli_test['sentence2_words'].apply(lambda x: ['']*max_len)

# 형태 변경
tmp = snli_test[['sentence1_words', 'sentence1_labels', 'sentence2_words', 'sentence2_labels']].stack().reset_index()
tmp_label_slots = pd.DataFrame(tmp[0].tolist(), index=tmp['level_1'])
tmp_ids = snli_test['pairID'].repeat(4)
labelling_template = pd.concat([tmp_ids.reset_index(drop=True), tmp_label_slots.reset_index()], axis=1)
assert labelling_template.shape[0] == 400
# 여태까지 형태
# pairID, level_1, 0, 1, 2, 3, ..., max_len-1
# ..., sentence1_words, ..., .., .., .., 
# ..., sentence1_labels, ,,,,
# ..., sentence2_words, ..., .., .., .., 
# ..., sentence2_labels, ,,,,

# 보기 좋게 sentence 쪼개기 전 문장도 추가 
adjusted_template = []
for ix, pairID in enumerate(snli_test['pairID']):
    curr_rows = labelling_template.loc[labelling_template['pairID']==pairID].reset_index(drop=True).copy()
    sentence1 = snli_test.loc[snli_test['pairID']==pairID, 'sentence1'].values[0]
    sentence2 = snli_test.loc[snli_test['pairID']==pairID, 'sentence2'].values[0]
    sentence1_df = pd.DataFrame({'pairID': [pairID], 
                                 'level_1': ['sentence1'],
                                 0: [sentence1]},index=[0])
    sentence2_df = pd.DataFrame({'pairID': [pairID], 
                                 'level_1': ['sentence2'],
                                 0: [sentence2]},index=[0])
    for i in range(1, 54):
        sentence1_df.loc[:, i] = ['']
        sentence2_df.loc[:, i] = ['']
    curr_rows = pd.concat([sentence1_df,sentence2_df, curr_rows], axis=0, ignore_index=True)
    curr_rows.index = [ix] * len(curr_rows)
    adjusted_template.append(curr_rows)
    
adjusted_template = pd.concat(adjusted_template, axis=0) 
# 여태까지 형태
# pairID, level_1, 0, 1, 2, 3, ..., max_len-1
# ..., sentence1, ..., ,,,,
# ..., sentence2, ...,,,,,
# ..., sentence1_words, ..., .., .., .., 
# ..., sentence1_labels, ,,,,
# ..., sentence2_words, ..., .., .., .., 
# ..., sentence2_labels, ,,,,
print(adjusted_template)
adjusted_template.to_excel(templated_save_path)

In [65]:
# 원본 데이터
snli_test = pd.read_json('/data/hyeryung/mucoco/data/nli/multinli_1.0/multinli_1.0_dev_matched.jsonl',lines=True)
snli_test2 = pd.read_json('/data/hyeryung/mucoco/data/nli/multinli_1.0/multinli_1.0_dev_mismatched.jsonl',lines=True)
snli_test = pd.concat([snli_test, snli_test2], axis=0)

# EPR 데이터
epr_snli = pd.read_json('/data/hyeryung/mucoco/new_module/data/EPR/text_file/mnli_annotation/mnli_matched_locate_labels.jsonl', lines=True)
# 학습 데이터
ours_train = pd.read_json('/data/hyeryung/mucoco/data/nli/snli_mnli_anli_train_dev_with_finegrained.jsonl', lines=True)
ours_train = ours_train.loc[ours_train['split']=='train'].copy()

# 저장 경로 1
raw_save_path = '/data/hyeryung/mucoco/new_module/data/NLI_locate/mnli_dev_contra_100_stratified_by_genre_seed_999.jsonl'
templated_save_path = '/data/hyeryung/mucoco/new_module/data/NLI_locate/mnli_dev_contra_labelling_sheet.xlsx'

In [66]:
# ---------------------- 라벨링 대상 추출, 저장 ---------------------- #
# 원본에서 EPR에 없는 데이터만 추출
print('original data', snli_test.shape[0])
snli_test = snli_test.loc[~snli_test['pairID'].isin(epr_snli['pairID']),:].copy()
print('after dropping EPR', snli_test.shape[0])
# 원본에서 training dataset에 없는 데이터만 추출
snli_test = snli_test.loc[~snli_test['pairID'].isin(ours_train['pairID']),:].copy()
print('after dropping train data', snli_test.shape[0])
# contradiction만 추출
snli_test = snli_test.loc[snli_test['gold_label']=='contradiction',:].copy()
print('after only selecting contradiction', snli_test.shape[0])
print(snli_test.gold_label.value_counts())
print(snli_test.annotator_labels.apply(Counter).value_counts())
print(snli_test.genre.value_counts())


# random 100개 추출
snli_test = snli_test.groupby('genre').sample(10, random_state=999)
print(snli_test.genre.value_counts())
# 저장 # !!!!!!!!!!!!!!
snli_test.to_json(raw_save_path, orient='records', lines=True)    
# double check
print(snli_test.gold_label.value_counts())

original data 20000
after dropping EPR 19940
after dropping train data 2306
after only selecting contradiction 637
gold_label
contradiction    637
Name: count, dtype: int64
annotator_labels
{'contradiction': 5}                                   461
{'contradiction': 4, 'neutral': 1}                      86
{'contradiction': 3, 'neutral': 2}                      47
{'contradiction': 4, 'entailment': 1}                   28
{'contradiction': 3, 'entailment': 1, 'neutral': 1}     10
{'contradiction': 3, 'entailment': 2}                    5
Name: count, dtype: int64
genre
verbatim      69
telephone     68
travel        67
letters       65
nineeleven    65
facetoface    65
government    62
oup           61
slate         59
fiction       56
Name: count, dtype: int64


In [44]:
# ---------------------- 라벨링 형식으로 바꿔서 저장 ---------------------- #
# labelling 하는 곳 추가
snli_test = snli_test[['pairID', 'sentence1','sentence2']].copy()
# text를 단어로 분리 
snli_test.loc[:, 'sentence1_words'] = snli_test['sentence1'].str.split()
snli_test.loc[:, 'sentence2_words'] = snli_test['sentence2'].str.split()
max_len =max(snli_test['sentence1_words'].apply(len).max(),snli_test['sentence2_words'].apply(len).max()) 
print(max_len)
# padding
snli_test.loc[:, 'sentence1_words'] = snli_test['sentence1_words'].apply(lambda x: x + ['']*(max_len-len(x)))
snli_test.loc[:, 'sentence2_words'] = snli_test['sentence2_words'].apply(lambda x: x + ['']*(max_len-len(x)))
assert min(snli_test['sentence1_words'].apply(len).min(),snli_test['sentence2_words'].apply(len).min()) == max_len
# labelling 할 곳 추가
snli_test.loc[:, 'sentence1_labels'] = snli_test['sentence1_words'].apply(lambda x: ['']*max_len)
snli_test.loc[:, 'sentence2_labels'] = snli_test['sentence2_words'].apply(lambda x: ['']*max_len)

# 형태 변경
tmp = snli_test[['sentence1_words', 'sentence1_labels', 'sentence2_words', 'sentence2_labels']].stack().reset_index()
tmp_label_slots = pd.DataFrame(tmp[0].tolist(), index=tmp['level_1'])
tmp_ids = snli_test['pairID'].repeat(4)
labelling_template = pd.concat([tmp_ids.reset_index(drop=True), tmp_label_slots.reset_index()], axis=1)
assert labelling_template.shape[0] == 400
# 여태까지 형태
# pairID, level_1, 0, 1, 2, 3, ..., max_len-1
# ..., sentence1_words, ..., .., .., .., 
# ..., sentence1_labels, ,,,,
# ..., sentence2_words, ..., .., .., .., 
# ..., sentence2_labels, ,,,,

# 보기 좋게 sentence 쪼개기 전 문장도 추가 
adjusted_template = []
for ix, pairID in enumerate(snli_test['pairID']):
    curr_rows = labelling_template.loc[labelling_template['pairID']==pairID].reset_index(drop=True).copy()
    sentence1 = snli_test.loc[snli_test['pairID']==pairID, 'sentence1'].values[0]
    sentence2 = snli_test.loc[snli_test['pairID']==pairID, 'sentence2'].values[0]
    sentence1_df = pd.DataFrame({'pairID': [pairID], 
                                 'level_1': ['sentence1'],
                                 0: [sentence1]},index=[0])
    sentence2_df = pd.DataFrame({'pairID': [pairID], 
                                 'level_1': ['sentence2'],
                                 0: [sentence2]},index=[0])
    for i in range(1, 54):
        sentence1_df.loc[:, i] = ['']
        sentence2_df.loc[:, i] = ['']
    curr_rows = pd.concat([sentence1_df,sentence2_df, curr_rows], axis=0, ignore_index=True)
    curr_rows.index = [ix] * len(curr_rows)
    adjusted_template.append(curr_rows)
    
adjusted_template = pd.concat(adjusted_template, axis=0) 
# 여태까지 형태
# pairID, level_1, 0, 1, 2, 3, ..., max_len-1
# ..., sentence1, ..., ,,,,
# ..., sentence2, ...,,,,,
# ..., sentence1_words, ..., .., .., .., 
# ..., sentence1_labels, ,,,,
# ..., sentence2_words, ..., .., .., .., 
# ..., sentence2_labels, ,,,,
print(adjusted_template)
adjusted_template.to_excel(templated_save_path)

53
     pairID           level_1  \
0   121660e         sentence1   
0   121660e         sentence2   
0   121660e   sentence1_words   
0   121660e  sentence1_labels   
0   121660e   sentence2_words   
..      ...               ...   
99  114240c         sentence2   
99  114240c   sentence1_words   
99  114240c  sentence1_labels   
99  114240c   sentence2_words   
99  114240c  sentence2_labels   

                                                    0        1           2  \
0                                    The whole thing?                        
0   In your childhood, reading what books was plea...                        
0                                                 The    whole      thing?   
0                                                                            
0                                                  In     your  childhood,   
..                                                ...      ...         ...   
99  Everyone dealing with 'dirty language' prefers... 

In [11]:
# 원본 데이터
snli_test = pd.read_json('/data/hyeryung/mucoco/data/nli/anli_v1.0/R1/test.jsonl',lines=True)
snli_test2 = pd.read_json('/data/hyeryung/mucoco/data/nli/anli_v1.0/R2/test.jsonl',lines=True)
snli_test3 = pd.read_json('/data/hyeryung/mucoco/data/nli/anli_v1.0/R3/test.jsonl',lines=True)
snli_test = pd.concat([snli_test, snli_test2, snli_test3], axis=0)

# 저장 경로 1
raw_save_path = '/data/hyeryung/mucoco/new_module/data/NLI_locate/anli_test_contra_100_stratified_by_genre_seed_999.jsonl'
templated_save_path = '/data/hyeryung/mucoco/new_module/data/NLI_locate/anli_test_contra_labelling_sheet.xlsx'

In [12]:
print(snli_test.columns)
snli_test['label'].unique()

Index(['uid', 'context', 'hypothesis', 'label', 'model_label', 'emturk',
       'genre', 'reason', 'tag'],
      dtype='object')


array(['e', 'n', 'c'], dtype=object)

In [13]:
# ---------------------- 라벨링 대상 추출, 저장 ---------------------- #
# contradiction만 추출
snli_test = snli_test.loc[snli_test['label']=='c',:].copy()
print('after only selecting contradiction', snli_test.shape[0])
print(snli_test.label.value_counts())

after only selecting contradiction 1062
label
c    1062
Name: count, dtype: int64


In [14]:
print(snli_test.genre.value_counts())

genre
wikipedia            732
news                  66
fiction               66
procedural/causal     66
legal/formal          66
rte                   66
Name: count, dtype: int64


In [15]:
snli_test.genre.value_counts() / len(snli_test)*100

genre
wikipedia            68.926554
news                  6.214689
fiction               6.214689
procedural/causal     6.214689
legal/formal          6.214689
rte                   6.214689
Name: count, dtype: float64

In [60]:
# 장르별로 원래 test에 있는 비율대로 추출 
snli_test_wiki = snli_test.loc[snli_test['genre']=='wikipedia',:].sample(70, random_state=999)
snli_test_others = snli_test.loc[snli_test['genre']!='wikipedia',:].groupby('genre').sample(6, random_state=999)

In [61]:
snli_test_wiki.genre.value_counts(), snli_test_others.genre.value_counts()

(genre
 wikipedia    70
 Name: count, dtype: int64,
 genre
 fiction              6
 legal/formal         6
 news                 6
 procedural/causal    6
 rte                  6
 Name: count, dtype: int64)

In [62]:
snli_test = pd.concat([snli_test_wiki, snli_test_others], axis=0)
print(snli_test.genre.value_counts())
# 저장 # !!!!!!!!!!!!!!
snli_test.to_json(raw_save_path, orient='records', lines=True)    
# double check
print(snli_test.label.value_counts())

genre
wikipedia            70
fiction               6
legal/formal          6
news                  6
procedural/causal     6
rte                   6
Name: count, dtype: int64
label
c    100
Name: count, dtype: int64


In [63]:
snli_test = snli_test.rename(columns={'uid': 'pairID', 'context': 'sentence1', 'hypothesis': 'sentence2'})

In [64]:
# ---------------------- 라벨링 형식으로 바꿔서 저장 ---------------------- #
# labelling 하는 곳 추가
snli_test = snli_test[['pairID', 'sentence1','sentence2']].copy()
# text를 단어로 분리 
snli_test.loc[:, 'sentence1_words'] = snli_test['sentence1'].str.split()
snli_test.loc[:, 'sentence2_words'] = snli_test['sentence2'].str.split()
max_len =max(snli_test['sentence1_words'].apply(len).max(),snli_test['sentence2_words'].apply(len).max()) 
print(max_len)
# padding
snli_test.loc[:, 'sentence1_words'] = snli_test['sentence1_words'].apply(lambda x: x + ['']*(max_len-len(x)))
snli_test.loc[:, 'sentence2_words'] = snli_test['sentence2_words'].apply(lambda x: x + ['']*(max_len-len(x)))
assert min(snli_test['sentence1_words'].apply(len).min(),snli_test['sentence2_words'].apply(len).min()) == max_len
# labelling 할 곳 추가
snli_test.loc[:, 'sentence1_labels'] = snli_test['sentence1_words'].apply(lambda x: ['']*max_len)
snli_test.loc[:, 'sentence2_labels'] = snli_test['sentence2_words'].apply(lambda x: ['']*max_len)

# 형태 변경
tmp = snli_test[['sentence1_words', 'sentence1_labels', 'sentence2_words', 'sentence2_labels']].stack().reset_index()
tmp_label_slots = pd.DataFrame(tmp[0].tolist(), index=tmp['level_1'])
tmp_ids = snli_test['pairID'].repeat(4)
labelling_template = pd.concat([tmp_ids.reset_index(drop=True), tmp_label_slots.reset_index()], axis=1)
assert labelling_template.shape[0] == 400
# 여태까지 형태
# pairID, level_1, 0, 1, 2, 3, ..., max_len-1
# ..., sentence1_words, ..., .., .., .., 
# ..., sentence1_labels, ,,,,
# ..., sentence2_words, ..., .., .., .., 
# ..., sentence2_labels, ,,,,

# 보기 좋게 sentence 쪼개기 전 문장도 추가 
adjusted_template = []
for ix, pairID in enumerate(snli_test['pairID']):
    curr_rows = labelling_template.loc[labelling_template['pairID']==pairID].reset_index(drop=True).copy()
    sentence1 = snli_test.loc[snli_test['pairID']==pairID, 'sentence1'].values[0]
    sentence2 = snli_test.loc[snli_test['pairID']==pairID, 'sentence2'].values[0]
    sentence1_df = pd.DataFrame({'pairID': [pairID], 
                                 'level_1': ['sentence1'],
                                 0: [sentence1]},index=[0])
    sentence2_df = pd.DataFrame({'pairID': [pairID], 
                                 'level_1': ['sentence2'],
                                 0: [sentence2]},index=[0])
    for i in range(1, 54):
        sentence1_df.loc[:, i] = ['']
        sentence2_df.loc[:, i] = ['']
    curr_rows = pd.concat([sentence1_df,sentence2_df, curr_rows], axis=0, ignore_index=True)
    curr_rows.index = [ix] * len(curr_rows)
    adjusted_template.append(curr_rows)
    
adjusted_template = pd.concat(adjusted_template, axis=0) 
# 여태까지 형태
# pairID, level_1, 0, 1, 2, 3, ..., max_len-1
# ..., sentence1, ..., ,,,,
# ..., sentence2, ...,,,,,
# ..., sentence1_words, ..., .., .., .., 
# ..., sentence1_labels, ,,,,
# ..., sentence2_words, ..., .., .., .., 
# ..., sentence2_labels, ,,,,
print(adjusted_template)
adjusted_template.to_excel(templated_save_path)

126
                                  pairID           level_1  \
0   b137bf6e-dfb0-4a39-81a4-8ddb08d4f206         sentence1   
0   b137bf6e-dfb0-4a39-81a4-8ddb08d4f206         sentence2   
0   b137bf6e-dfb0-4a39-81a4-8ddb08d4f206   sentence1_words   
0   b137bf6e-dfb0-4a39-81a4-8ddb08d4f206  sentence1_labels   
0   b137bf6e-dfb0-4a39-81a4-8ddb08d4f206   sentence2_words   
..                                   ...               ...   
99  bd1ef158-7c54-4bba-80f3-d7f18d95a94f         sentence2   
99  bd1ef158-7c54-4bba-80f3-d7f18d95a94f   sentence1_words   
99  bd1ef158-7c54-4bba-80f3-d7f18d95a94f  sentence1_labels   
99  bd1ef158-7c54-4bba-80f3-d7f18d95a94f   sentence2_words   
99  bd1ef158-7c54-4bba-80f3-d7f18d95a94f  sentence2_labels   

                                                    0       1         2     3  \
0   Ai Weiwei The Fake Case is a 2013 documentary ...                           
0   The film won Best Documentary in Danish Film C...                           
0       

## Export data for label studio

In [ ]:
import pandas as pd

anli = pd.read_json('/data/hyeryung/mucoco/new_module/data/NLI_locate/anli_test_contra_100_stratified_by_genre_seed_999.jsonl', lines=True)
anli = anli.rename(columns={'uid': 'pairID', 'context': 'sentence1', 'hypothesis': 'sentence2'})
anli['source'] = 'anli'
# anli[['pairID','source','sentence1','sentence2']].to_csv('/data/hyeryung/mucoco/new_module/data/NLI_locate/anli_test_contra_100_stratified_by_genre_seed_999.csv', index=False)

In [20]:

mnli = pd.read_json('/data/hyeryung/mucoco/new_module/data/NLI_locate/mnli_dev_contra_100_stratified_by_genre_seed_999.jsonl', lines=True)
mnli = mnli.rename(columns={'uid': 'pairID', 'context': 'sentence1', 'hypothesis': 'sentence2'})
mnli['source'] = 'mnli'
# mnli[['pairID','source','sentence1','sentence2']].to_csv('/data/hyeryung/mucoco/new_module/data/NLI_locate/mnli_dev_contra_100_stratified_by_genre_seed_999.csv', index=False)

In [21]:
snli = pd.read_json('/data/hyeryung/mucoco/new_module/data/NLI_locate/snli_test_contra_100_seed_999.jsonl', lines=True)
snli = snli.rename(columns={'uid': 'pairID', 'context': 'sentence1', 'hypothesis': 'sentence2'})
snli['source'] = 'snli'
# snli[['pairID','source','sentence1','sentence2']].to_csv('/data/hyeryung/mucoco/new_module/data/NLI_locate/snli_test_contra_100_seed_999.csv', index=False)

In [23]:
pd.concat([anli[['pairID','source','sentence1','sentence2']], mnli[['pairID','source','sentence1','sentence2']], snli[['pairID','source','sentence1','sentence2']]], axis=0).to_csv('/data/hyeryung/mucoco/new_module/data/NLI_locate/nli_contra_300.csv', index=False)